# Credit Risk Guru — authored entirely in natural language, with a reasoning audit trail

Everything this notebook feeds the runtime is **plain English**. There is not one
hardwired function, scoring rule, decision grid or `def` anywhere below: the whole
stack is six markdown files written in natural language, and `load_runtime` assembles
the Enterprise Agentic Runtime from them —

```text
prompts   stacked in  skills.md
skills    stacked in  persona.md
steps     stacked in  workflow.md
workflows stacked in  process.md
governance, risk & controls stacked in  policy.md
strategy (context history, cross-session data, subagent spawning, model
selection, audit trail, ontology) stacked in  memory.md
```

And every judgment the LLM makes — each policy verdict with its rationale, the
process discovery, the deliberation with the *full stacked prompt material it
reasoned over*, and the explanation — is written to a **ReasoningLog audit trail**
(in memory and as append-only JSONL), so reasoning is reviewable after the fact
and the stacked prompts can be optimised against what the model actually saw.

The Anthropic credential is read **only** from the `ANTHROPIC_API_KEY` environment
variable, exactly as `memory.md` declares. Without it, every stage degrades to its
deterministic fallback — and the audit trail still records every step.

In [1]:
import json
import os
import tempfile
from pathlib import Path

from ear import Intent, load_runtime

stack = Path(tempfile.mkdtemp(prefix="ear_credit_risk_")) / "credit_risk_stack"
stack.mkdir()

print("Stack directory:", stack)
print(
    "ANTHROPIC_API_KEY is set -- reasoning runs against a live model"
    if os.environ.get("ANTHROPIC_API_KEY")
    else "No ANTHROPIC_API_KEY -- deterministic fallbacks run, and the trail still records every step"
)

Stack directory: /tmp/ear_credit_risk_wqtt3srt/credit_risk_stack
ANTHROPIC_API_KEY is set -- reasoning runs against a live model


## 1. Prompts stacked in `skills.md`

Each heading names a skill; the prose beneath it **is** the prompt the runtime
reasons with. The banding, grading and decision rules live in these sentences —
there is no scoring function anywhere.

In [2]:
(stack / "skills.md").write_text("""# Skills

## band_credit_profile

Read the applicant's credit score, income and existing debt from the intent's
context. Band the credit score into poor, fair, good or excellent, and the
debt-to-income ratio into low, moderate or high.

## assign_risk_grade

Combine the score band and the debt-to-income band into a single risk grade
from A (strongest) to E (weakest), and say in one sentence why that grade
follows from the bands.

## decide_application

Given the risk grade, decide approve or decline. Approve grades A to C when
no policy is violated; decline grades D and E, naming the decisive factor.

## write_customer_note

Draft a short, courteous note to the applicant stating the decision and its
main reason, in plain English with no internal jargon.
""")
print("skills.md stacked -- four prompts, zero functions")

skills.md stacked -- four prompts, zero functions


## 2. Skills stacked in `persona.md`

The prose is the persona's standing instructions; the `Skills:` line stacks
skills from `skills.md` by name.

In [3]:
(stack / "persona.md").write_text("""# Personas

## Credit Risk Guru

Underwrite conservatively. Prefer declining a marginal application over
approving a risky one, and always name the decisive factor behind a judgment.

Skills: band_credit_profile, assign_risk_grade, decide_application

## Customer Advocate

Speak plainly and kindly to applicants. Never expose internal risk jargon or
scores; explain decisions in terms an applicant can act on.

Skills: write_customer_note
""")
print("persona.md stacked -- two personas")

persona.md stacked -- two personas


## 3. Steps stacked in `workflow.md`

Numbered steps, narrated in plain English, each delegated to the persona named
in parentheses. `Policies:` attaches governance from `policy.md` by name. This
is the only control flow that exists — and it is prose.

In [4]:
(stack / "workflow.md").write_text("""# Workflows

## Underwriting Workflow

1. Band the applicant's credit profile. (Credit Risk Guru)
2. Assign a risk grade from the banded profile. (Credit Risk Guru)
3. Decide approve or decline against the grade. (Credit Risk Guru)
4. Write the customer note announcing the decision. (Customer Advocate)

Policies: Loan Amount Cap
""")
print("workflow.md stacked -- four delegated steps")

workflow.md stacked -- four delegated steps


## 4. Workflows stacked in `process.md`

The prose is the description the `Discoverer` reasons over when matching an
intent to a process; the file's `#` title names the runtime itself.

In [5]:
(stack / "process.md").write_text("""# Credit Risk Guru Runtime

## Underwrite Consumer Loan

Evaluates a consumer loan application end to end: banding the credit profile,
assigning a risk grade, deciding approve or decline, and writing the customer
note.

Workflows: Underwriting Workflow
""")
print("process.md stacked -- one process, and the runtime is named")

process.md stacked -- one process, and the runtime is named


## 5. Governance, risk and controls stacked in `policy.md`

Each statement is judged **by the LLM in natural language** against the intent's
context. `Fallback:` keeps the same policy enforceable deterministically when no
model is configured, and `Applies to:` maps it onto the runtime or a workflow.

In [6]:
(stack / "policy.md").write_text("""# Policies

## Loan Amount Cap

The requested loan amount must not exceed $75,000.

Fallback: loan_amount <= 75000
Applies to: Underwriting Workflow

## Debt-to-Income Ceiling

The applicant's debt-to-income ratio must not exceed 0.43.

Fallback: debt_to_income <= 0.43
Applies to: runtime

## Fair Lending Control

Decisions must rest only on the financial facts carried in the intent's
context -- never on age, gender, ethnicity or any other protected attribute.

Applies to: runtime
""")
print("policy.md stacked -- governance at two scopes")

policy.md stacked -- governance at two scopes


## 6. Strategy stacked in `memory.md`

Context history, cross-session data, the **reasoning audit trail**, subagent
spawning, model selection and the ontology — every setting below is read out of
the prose. The credential is named, never written: only the environment-variable
name appears in the file.

In [7]:
(stack / "memory.md").write_text("""# Memory & Strategy

## Context History

Keep the 20 most recent cycles verbatim; compress anything older into
summaries so the reasoning context stays bounded.

## Cross-Session Data

Persist memory, experience and learned adaptations to `.ear/session.json` so
a new session picks up exactly where the last one left off.

## Reasoning Audit Trail

Log every reasoning step -- each policy judgment with its rationale, process
discovery, the deliberation with the full stacked prompt material, and the
explanation -- to `.ear/reasoning.jsonl`, append-only, so the trail can be
reviewed and the stacked prompts optimised.

## Subagent Spawning

Allow spawning up to 4 subagents, each scoped to a single persona.

## Model Selection

Reason with anthropic/claude-haiku-4-5, reading the credential from
ANTHROPIC_API_KEY, at a temperature of 0.2. When the credential is absent
the runtime stays on its deterministic fallback.

## Ontological Settings

- risk grade: a letter from A to E, where A is the strongest credit and E the weakest
- debt-to-income: monthly debt obligations divided by gross monthly income
- decision: exactly one of approve or decline, never a hedge
""")
print("memory.md stacked -- the operating strategy, in prose")

memory.md stacked -- the operating strategy, in prose


## 7. Load the runtime from the stack

`load_runtime` resolves every cross-reference by name (failing loudly if one is
wrong), maps the policies onto their scopes, and applies the strategy: memory
capacity, session store, spawner limits, model binding, and the audit trail path.

In [8]:
runtime = load_runtime(stack)

print("Runtime: ", runtime.name)
print("Model:   ", runtime.model_binding.model_id if runtime.model_binding else "deterministic fallback (no credential in the environment)")
print("Audit:   ", runtime.reasoning_log.path)
print("Sessions:", runtime.session_store.path)
print()
for process in runtime.processes:
    print(f"Process '{process.name}'")
    for workflow in process.workflows:
        print(f"  Workflow '{workflow.name}' -- governed by {[p.name for p in workflow.policies]}")
        for number, step in enumerate(workflow.steps, start=1):
            print(f"    {number}. {step.instruction}  ->  {step.persona.name}")
print("Runtime-wide policies:", [p.name for p in runtime.policies])

Runtime:  Credit Risk Guru Runtime
Model:    anthropic/claude-haiku-4-5
Audit:    /tmp/ear_credit_risk_wqtt3srt/credit_risk_stack/.ear/reasoning.jsonl
Sessions: /tmp/ear_credit_risk_wqtt3srt/credit_risk_stack/.ear/session.json

Process 'Underwrite Consumer Loan'
  Workflow 'Underwriting Workflow' -- governed by ['Loan Amount Cap']
    1. Band the applicant's credit profile.  ->  Credit Risk Guru
    2. Assign a risk grade from the banded profile.  ->  Credit Risk Guru
    3. Decide approve or decline against the grade.  ->  Credit Risk Guru
    4. Write the customer note announcing the decision.  ->  Customer Advocate
Runtime-wide policies: ['Debt-to-Income Ceiling', 'Fair Lending Control']


## 8. Reason an application through the stack

The intent is a sentence; the context carries the applicant's *facts* only — no
tiers, bands or grades are precomputed anywhere. Banding, grading, the decision
and the customer note all happen inside reasoning, through the stacked prompts.

In [9]:
decision = runtime.reason(Intent(
    text="Underwrite a $18,500 personal loan for Priya Raman, requested for a kitchen renovation.",
    context={
        "loan_amount": 18500,
        "credit_score": 742,
        "debt_to_income": 0.28,
        "annual_income": 95000,
        "existing_defaults": 0,
    },
))
print(decision)

**APPROVE** the $18,500 personal loan for Priya Raman.

**Reasoning:**

*Step 1 & 2: Credit Profile Banding and Risk Grade Assignment*
- **Credit Score Band:** 742 is in the "good" range (typically 670-739 is fair, 740+ is good)
- **Debt-to-Income Band:** 0.28 (28%) is "low" (generally acceptable thresholds are under 36-43%)
- **Risk Grade:** B (good credit + low debt-to-income = strong profile). The applicant demonstrates solid creditworthiness with manageable debt obligations relative to income.

*Step 3: Decision Against Grade*
- Risk Grade B falls within the approve range (A-C approved when no policy violations exist)
- No policy violations present: no existing defaults, income supports the loan amount, debt-to-income is healthy
- **Decision: APPROVE**
- **Decisive Factor:** Strong credit score combined with low debt-to-income ratio indicates low credit risk and strong repayment capacity.

*Step 4: Customer Note*

Dear Priya,

Good news! We're pleased to approve your $18,500 person

## 9. Governance blocks before reasoning — and the block is on the record

An application breaching the Loan Amount Cap never reaches deliberation:
`PermissionError` is raised, no memory entry is written, and the violated
judgment — **with its rationale** — is still flushed to the audit trail.

In [10]:
try:
    runtime.reason(Intent(
        text="Underwrite a $120,000 personal loan for Arun Mehta.",
        context={"loan_amount": 120000, "credit_score": 810, "debt_to_income": 0.15},
    ))
except PermissionError as blocked:
    print("Blocked before deliberation:", blocked)

Blocked before deliberation: Workflow policy violated: Loan Amount Cap


## 10. The reasoning audit trail

Every judgment of both cycles — the approval and the block — one record per
stage, each attributed to the model that made it.

In [11]:
print(runtime.reasoning_log.render())

=== Cycle 1 (2026-07-02 02:05:55) ===
[intent] -> Underwrite a $18,500 personal loan for Priya Raman, requested for a kitchen renovation.
[policy] -> complies  (anthropic/claude-haiku-4-5)
    why: The applicant's debt-to-income ratio of 0.28 is below the maximum threshold of 0.43, so the policy requirement is satisfied.
[policy] -> complies  (anthropic/claude-haiku-4-5)
    why: The context contains only financial facts (loan amount, credit score, debt-to-income ratio, annual income, and defaults) with no reference to protected attri...
[discovery] -> Underwrite Consumer Loan  (anthropic/claude-haiku-4-5)
[policy] -> complies  (anthropic/claude-haiku-4-5)
    why: The requested loan amount of $18,500 is well below the maximum threshold of $75,000, so the context complies with the policy.
[deliberation] -> **APPROVE** the $18,500 personal loan for Priya Raman. **Reasoning:** *Step 1 & 2: Credit Profile Banding and Risk Grade Assignment* - **Credit Score Band:**...  (anthropic/claude-ha

## 11. Review exactly what the model reasoned with

This is the prompt-review loop. The `deliberation` record carries the full
stacked capabilities block — the narrated steps, the delegated personas, the
skill prompts — precisely as the Reasoner composed them. The `policy` records
carry each verdict's rationale. This is the material you read before deciding
which stacked prompt to sharpen.

In [12]:
deliberation = runtime.reasoning_log.for_stage("deliberation")[-1]
print("Model:", deliberation.model)
print()
print("--- The stacked prompt material the model reasoned with ---")
print(deliberation.inputs["capabilities"])
print()
print("--- Every policy judgment, with its rationale ---")
for record in runtime.reasoning_log.for_stage("policy"):
    print(f"cycle {record.cycle} | {record.inputs['policy']}: {record.output}")
    print(f"    why: {record.rationale}")

Model: anthropic/claude-haiku-4-5

--- The stacked prompt material the model reasoned with ---
Workflow Underwriting Workflow:
  Step 1: Band the applicant's credit profile. [delegated to Persona Credit Risk Guru]
      Persona Credit Risk Guru: Underwrite conservatively. Prefer declining a marginal application over approving a risky one, and always name the decisive factor behind a judgment.
        - Skill band_credit_profile: Read the applicant's credit score, income and existing debt from the intent's context. Band the credit score into poor, fair, good or excellent, and the debt-to-income ratio into low, moderate or high.
        - Skill assign_risk_grade: Combine the score band and the debt-to-income band into a single risk grade from A (strongest) to E (weakest), and say in one sentence why that grade follows from the bands.
        - Skill decide_application: Given the risk grade, decide approve or decline. Approve grades A to C when no policy is violated; decline grades D and 

## 12. The same trail, on disk as append-only JSONL

Declared in `memory.md`, flushed after every cycle (blocked ones included),
accumulating across sessions — ready for review tooling, or for curating a
GEPA trainset.

In [13]:
trail = Path(runtime.reasoning_log.path)
lines = trail.read_text(encoding="utf-8").splitlines()
print(f"{trail}\n{len(lines)} records on disk\n")
print("Last record:")
print(json.dumps(json.loads(lines[-1]), indent=2)[:900])

/tmp/ear_credit_risk_wqtt3srt/credit_risk_stack/.ear/reasoning.jsonl
12 records on disk

Last record:
{
  "cycle": 2,
  "stage": "policy",
  "timestamp": "2026-07-02T02:06:13.974990+00:00",
  "model": "anthropic/claude-haiku-4-5",
  "inputs": {
    "policy": "Loan Amount Cap",
    "statement": "The requested loan amount must not exceed $75,000.",
    "context": {
      "loan_amount": 120000,
      "credit_score": 810,
      "debt_to_income": 0.15
    }
  },
  "output": "VIOLATED",
  "rationale": "The requested loan amount of $120,000 exceeds the maximum allowed limit of $75,000, violating the policy."
}


## 13. Optimise a prompt from the trail — in natural language

Reviewing the deliberation records above, `decide_application` is vague about
marginal grade-C applicants. So we sharpen **the sentence** — not code — rewrite
`skills.md`, and reload. Cross-session data means the new session restores the
prior memory; the audit trail keeps accumulating in the same file, so before/after
behaviour of the reworded prompt is directly comparable.

In [14]:
(stack / "skills.md").write_text("""# Skills

## band_credit_profile

Read the applicant's credit score, income and existing debt from the intent's
context. Band the credit score into poor, fair, good or excellent, and the
debt-to-income ratio into low, moderate or high.

## assign_risk_grade

Combine the score band and the debt-to-income band into a single risk grade
from A (strongest) to E (weakest), and say in one sentence why that grade
follows from the bands.

## decide_application

Given the risk grade, decide approve or decline. Approve grades A and B
outright. For grade C, approve only when the debt-to-income band is low and
there are no existing defaults; otherwise decline. Decline grades D and E.
Always name the single decisive factor in one sentence.

## write_customer_note

Draft a short, courteous note to the applicant stating the decision and its
main reason, in plain English with no internal jargon.
""")

runtime = load_runtime(stack)
print(f"Reloaded. {len(runtime.memory.working)} prior cycle(s) restored from .ear/session.json")

decision = runtime.reason(Intent(
    text="Underwrite a $22,000 personal loan for Meera Iyer, a borderline applicant.",
    context={
        "loan_amount": 22000,
        "credit_score": 668,
        "debt_to_income": 0.41,
        "annual_income": 61000,
        "existing_defaults": 0,
    },
))
print()
print(decision)

Reloaded. 1 prior cycle(s) restored from .ear/session.json



**DECLINE** the $22,000 personal loan for Meera Iyer.

**Reasoning:**

*Step 1 & 2: Credit Profile Banding and Risk Grade Assignment*
- **Credit Score Band:** 668 is in the "fair" range (typically 670-739 is fair; 668 falls just below this threshold, approaching poor)
- **Debt-to-Income Band:** 0.41 (41%) is "moderate to high" (generally acceptable thresholds are under 36-43%, and 41% is at the upper boundary)
- **Risk Grade:** C (fair/borderline credit + moderate-to-high debt-to-income = marginal profile). The applicant demonstrates acceptable but constrained creditworthiness with elevated debt obligations relative to income.

*Step 3: Decision Against Grade*
- Risk Grade C requires approval only when debt-to-income is low AND there are no existing defaults
- Meera's debt-to-income is 0.41 (moderate-to-high), NOT low
- While there are no existing defaults, the elevated debt-to-income ratio disqualifies approval under the C-grade decision rule
- **Decision: DECLINE**
- **Decisive Fact

## 14. The new cycle in the trail — the optimisation is auditable too

In [15]:
print(runtime.reasoning_log.render(cycle=runtime.reasoning_log.cycle))
print()
sharpened = runtime.reasoning_log.for_stage("deliberation")[-1]
print("The sharpened prompt the model reasoned with this time:")
for line in sharpened.inputs["capabilities"].splitlines():
    if "decide_application" in line:
        print(" ", line.strip())

=== Cycle 1 (2026-07-02 02:06:14) ===
[intent] -> Underwrite a $22,000 personal loan for Meera Iyer, a borderline applicant.
[policy] -> complies  (anthropic/claude-haiku-4-5)
    why: The applicant's debt-to-income ratio of 0.41 is below the maximum threshold of 0.43, so the policy requirement is satisfied.
[policy] -> complies  (anthropic/claude-haiku-4-5)
    why: The context contains only financial facts (loan amount, credit score, debt-to-income ratio, annual income, and defaults) with no reference to age, gender, et...
[discovery] -> Underwrite Consumer Loan  (anthropic/claude-haiku-4-5)
[policy] -> complies  (anthropic/claude-haiku-4-5)
    why: The requested loan amount of $22,000 is well below the maximum threshold of $75,000, so the context complies with the policy.
[deliberation] -> **DECLINE** the $22,000 personal loan for Meera Iyer. **Reasoning:** *Step 1 & 2: Credit Profile Banding and Risk Grade Assignment* - **Credit Score Band:** ...  (anthropic/claude-haiku-4-5)
[exp

## Why this closes the loop

- **Only natural language went in.** Six markdown files: prompts stacked in
  skills.md, skills in persona.md, steps in workflow.md, workflows in
  process.md, governance in policy.md, strategy in memory.md. No function,
  handler, decision table or boolean was written in this notebook.
- **Every judgment came out on the record.** The ReasoningLog holds one record
  per stage per cycle — verdicts with rationale, discovery, the deliberation
  with the exact stacked prompt material, the explanation — in memory and as
  append-only JSONL declared in memory.md.
- **The trail drives prompt optimisation.** Review what the model actually
  reasoned with, reword a stacked prompt in English, reload, and compare
  cycles in the same trail. For reflective, automated optimisation of the
  Reasoner's own program, curate examples from the JSONL and call
  `runtime.reasoner.optimize_with_gepa(trainset, metric)` — GEPA is the one
  sanctioned optimizer in the package, kept on the single most judgment-laden
  stage.